<a href="https://colab.research.google.com/github/JaydenTran04/AAI2026/blob/main/Prompt_Chaining_for_a_Customer_Support_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import re

def classify_and_respond(customer_message):
    """
    BayBites Meal Kits support classifier and responder.
    Returns JSON with intent, clarify, and answer fields.
    """
    msg_lower = customer_message.lower()

    # Intent classification (new scenario)
    intent = None
    if any(word in msg_lower for word in ['refund', 'return', 'money back', 'chargeback', 'wrong charge']):
        intent = 'billing_refund'
    elif any(word in msg_lower for word in ['late', 'delivery', 'missing', 'where is', 'arrived', 'tracking', 'dropped off']):
        intent = 'delivery'
    elif any(word in msg_lower for word in ['allergy', 'ingredient', 'nutrition', 'calories', 'gluten', 'dairy', 'vegan']):
        intent = 'dietary'
    elif any(word in msg_lower for word in ['cancel', 'pause', 'skip', 'subscription', 'plan']):
        intent = 'subscription'
    else:
        # Default to delivery for ambiguous cases (common for meal kit support)
        intent = 'delivery'

    # Check for order/box ID (patterns: #A1B2C, box 12345, order ID: XYZ12)
    box_id_match = re.search(r'(?:#|box\s*|order\s*id\s*|order\s*)[:\s]*([A-Z0-9]{5,})',
                             customer_message, re.IGNORECASE)
    has_box_id = box_id_match is not None

    # Check for location/time detail (useful for delivery issues)
    delivery_clues = ['today', 'yesterday', 'tonight', 'morning', 'front door', 'lobby', 'gate', 'apartment', 'unit']
    has_delivery_detail = any(word in msg_lower for word in delivery_clues) and len(customer_message.split()) > 5

    # Determine if clarification is needed
    clarify = "NONE"
    answer = ""

    if intent in ['billing_refund']:
        if not has_box_id:
            clarify = "Can you share your order/box ID (or the email used for the purchase)?"
            answer = "PENDING_INFO"
        else:
            answer = (
                "Thanks — I can help with that. I’ve submitted a refund review for your order. "
                "You’ll get an email confirmation within 24 hours with the refund timeline."
            )

    elif intent == 'delivery':
        if not has_box_id:
            clarify = "What’s your order/box ID so I can check the delivery status?"
            answer = "PENDING_INFO"
        elif not has_delivery_detail:
            clarify = "When was it supposed to arrive, and where do you usually receive deliveries (front door/lobby/etc.)?"
            answer = "PENDING_INFO"
        else:
            answer = (
                "I found your delivery details. If it’s marked delivered but you don’t see it, "
                "please check nearby drop-off spots (lobby/front desk/side door) and confirm your address on file. "
                "If it’s still missing, I can start a replacement request."
            )

    elif intent == 'dietary':
        clarify = "Which meal name are you asking about, and what ingredient(s) are you avoiding?"
        answer = "PENDING_INFO"

    else:  # subscription
        if any(word in msg_lower for word in ['cancel']):
            answer = (
                "I can help you cancel. To confirm, do you want to cancel immediately or after your next scheduled box?"
            )
        elif any(word in msg_lower for word in ['pause', 'skip']):
            answer = (
                "You can pause or skip a week from your account page under Plan Settings. "
                "Tell me the week/date you want to skip and I’ll guide you."
            )
        else:
            clarify = "Are you trying to cancel, pause, or change your plan?"
            answer = "PENDING_INFO"

    return json.dumps({
        "intent": intent,
        "clarify": clarify,
        "answer": answer
    }, indent=2)


# Demo examples (new scenario)
print("=" * 60)
print("DEMO 1: Refund request with order ID")
print("=" * 60)
customer_message = "I need a refund for order #BX91Q"
print(f"Customer: {customer_message}\n")
print(classify_and_respond(customer_message))

print("\n" + "=" * 60)
print("DEMO 2: Refund request WITHOUT order ID")
print("=" * 60)
customer_message = "I was charged twice for my meal kit"
print(f"Customer: {customer_message}\n")
print(classify_and_respond(customer_message))

print("\n" + "=" * 60)
print("DEMO 3: Delivery inquiry with order ID + detail")
print("=" * 60)
customer_message = "Where is my box? Order ID: QZ7721. It was supposed to arrive today to my apartment lobby."
print(f"Customer: {customer_message}\n")
print(classify_and_respond(customer_message))

print("\n" + "=" * 60)
print("DEMO 4: Dietary question")
print("=" * 60)
customer_message = "Does the Thai peanut bowl contain dairy? I have an allergy."
print(f"Customer: {customer_message}\n")
print(classify_and_respond(customer_message))

print("\n" + "=" * 60)
print("DEMO 5: Subscription change")
print("=" * 60)
customer_message = "I want to skip next week’s delivery"
print(f"Customer: {customer_message}\n")
print(classify_and_respond(customer_message))

DEMO 1: Refund request with order ID
Customer: I need a refund for order #BX91Q

{
  "intent": "billing_refund",
  "clarify": "NONE",
  "answer": "Thanks \u2014 I can help with that. I\u2019ve submitted a refund review for your order. You\u2019ll get an email confirmation within 24 hours with the refund timeline."
}

DEMO 2: Refund request WITHOUT order ID
Customer: I was charged twice for my meal kit

{
  "intent": "delivery",
  "clarify": "What\u2019s your order/box ID so I can check the delivery status?",
  "answer": "PENDING_INFO"
}

DEMO 3: Delivery inquiry with order ID + detail
Customer: Where is my box? Order ID: QZ7721. It was supposed to arrive today to my apartment lobby.

{
  "intent": "delivery",
  "clarify": "NONE",
  "answer": "I found your delivery details. If it\u2019s marked delivered but you don\u2019t see it, please check nearby drop-off spots (lobby/front desk/side door) and confirm your address on file. If it\u2019s still missing, I can start a replacement request